# 05 — Détection YOLO et deux étages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/05_detection.ipynb)

Phase 2 (docs/04-detection-yolo.md) : jeu YOLO `data/yolo/` (PlantDoc + PlantVillage,
classe `feuille`), entraînement du détecteur suivi dans MLflow, puis pipeline deux étages
(YOLO → crops → classifieur de la phase 1) évalué de bout en bout sur PlantDoc TEST.
`YOLO_RUN` renseigné : pas d'entraînement, le run existant est évalué.

Sur Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## Paramètres

In [ ]:
CONFIG = "configs/yolo_v8n.yaml"
SET = []  # surcharges, ex. ["yolo.epochs=50", "model=yolov8s.pt"]
DEVICE = None  # None : Ultralytics choisit (GPU 0 s'il y en a un)
YOLO_RUN = None  # run existant (runs/<id>/best.pt) : saute l'entraînement
PV_PER_CLASS = 40  # feuilles PlantVillage par classe dans le jeu YOLO
REBUILD = False  # reconstruit data/yolo/ même s'il existe
CLF_RUN = None  # classifieur de la phase 1 pour les deux étages ; None : le plus récent
LIMIT = 0  # images PlantDoc TEST évaluées (0 : toutes)
DATASETS = ["index", "plantvillage", "plantdoc"]

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision,detection]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

In [ ]:
# Entraînement, évaluation et Grad-CAM tournent sur le GPU, augmentations comprises.
import torch

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
elif COLAB:
    raise RuntimeError("pas de GPU : Exécution → Modifier le type d'exécution → GPU (T4)")
else:
    print("pas de GPU : tout tournera sur CPU (lent)")

## Jeu YOLO `data/yolo/`

In [ ]:
from IPython.display import Image, display

from vision.datasets.build_index import DATA_DIR
from vision.labeling import build_yolo_dataset as byd

YOLO_DIR = DATA_DIR / byd.OUT_NAME
if REBUILD or not (YOLO_DIR / byd.MANIFEST).exists():
    byd.build(DATA_DIR, YOLO_DIR, PV_PER_CLASS, os.cpu_count())

In [ ]:
import pandas as pd

manifest = pd.read_parquet(YOLO_DIR / byd.MANIFEST)
byd.report(manifest)
byd.preview(YOLO_DIR, "train", path=YOLO_DIR / "preview_train.png")
display(Image(filename=YOLO_DIR / "preview_train.png"))

## Configuration

In [ ]:
import yaml

from vision.train.config import apply_overrides, load_config

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
YOLO_ENV_SET = (
    [f"yolo.workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)
cfg = apply_overrides(load_config(ROOT / CONFIG), YOLO_ENV_SET + SET)
print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

## Entraînement

In [ ]:
from vision.train.train_yolo import run

if YOLO_RUN is None:
    with drive_data.PeriodicSync(drive_data.OUTPUTS, DRIVE_DIR):
        YOLO_RUN = run(cfg, DEVICE).name
print("YOLO_RUN =", YOLO_RUN)

## Courbes et métriques (val, puis test)

In [ ]:
UL = ROOT / "runs" / "ultralytics"
for png in ["results.png", "BoxPR_curve.png"]:
    for d in [UL / YOLO_RUN, UL / f"{YOLO_RUN}_test"]:
        if (d / png).exists():
            print(d.name, png)
            display(Image(filename=d / png))

## Deux étages sur PlantDoc TEST

In [ ]:
import json

from vision.eval.two_stage import (
    CropClassifier,
    TwoStagePipeline,
    YoloDetector,
    evaluate,
    plantdoc_scenes,
    plot_examples,
)
from vision.eval.cross_domain_eval import REPORTS_DIR
from vision.train.train_classifier import pick_device

if CLF_RUN is None:  # the YOLO runs have a runs/ultralytics/<id>/ folder
    ckpts = [p for p in (ROOT / "runs").glob("*/best.pt") if not (UL / p.parent.name).exists()]
    if not ckpts:
        raise FileNotFoundError("aucun classifieur runs/*/best.pt : notebook 02")
    CLF_RUN = max(ckpts, key=lambda p: p.stat().st_mtime).parent.name
print("CLF_RUN =", CLF_RUN)

device = pick_device(DEVICE)
pipeline = TwoStagePipeline(
    YoloDetector(ROOT / "runs" / YOLO_RUN / "best.pt", device=str(device)),
    CropClassifier(ROOT / "runs" / CLF_RUN / "best.pt", device),
)
scenes = plantdoc_scenes(DATA_DIR, "test")
scenes = scenes[:LIMIT] if LIMIT else scenes
metrics, results = evaluate(pipeline, scenes)
OUT = REPORTS_DIR / CLF_RUN
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "two_stage_test.json").write_text(json.dumps(metrics, indent=1))
plot_examples(scenes, results, OUT / "two_stage_test.png")
drive_data.sync_all(DRIVE_DIR)
display(pd.Series(metrics).to_frame("valeur"))
display(Image(filename=OUT / "two_stage_test.png"))

## Résumé

Détecteur : `runs/<YOLO_RUN>/best.pt` (courbes dans `runs/ultralytics/<YOLO_RUN>/`) ;
deux étages : `reports/<CLF_RUN>/two_stage_test.{json,png}`. Critère de la feuille de
route (docs/07) : mAP@0.5 « feuille » ≥ 0,6. Suite : pseudo-labels SAM 2 et Label
Studio (`python -m vision.labeling.pseudo_labels`, docs/04 §2).